# W08: 고객 세그먼트 분석

주문 데이터 기반으로 RFM을 계산해 VIP/충성/이탈우려/새고객으로 분류하고 세그먼트를 시각화합니다.


In [ ]:
import io
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for fp in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(fp)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}
    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        result = model.generate_content(prompt)
        return getattr(result, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

from datetime import datetime

uploaded = safe_upload()
if uploaded:
    fn = list(uploaded.keys())[0]
    od = pd.read_csv(io.StringIO(uploaded[fn].decode("utf-8")))
else:
    od = pd.DataFrame(
        {
            "고객ID": ["U1", "U1", "U2", "U3", "U3", "U4", "U5", "U5", "U5", "U6"],
            "주문일": ["2026-01-01", "2026-01-12", "2026-01-03", "2026-01-05", "2026-01-30", "2026-01-02", "2026-01-15", "2026-01-20", "2026-01-27", "2026-01-10"],
            "주문금액": [12000, 18000, 7000, 25000, 14000, 8000, 32000, 11000, 9000, 5000]
        }
    )

od["주문일"] = pd.to_datetime(od["주문일"])
od = od.sort_values("주문일")
last_date = od["주문일"].max()
rfm = od.groupby("고객ID").agg(
    최근구매일=("주문일", "max"),
    구매횟수=("주문금액", "count"),
    총매출=("주문금액", "sum")
).reset_index()
rfm["Recency"] = (last_date - rfm["최근구매일"]).dt.days
rfm["RecencyScore"] = pd.qcut(rfm["Recency"], 4, labels=[4,3,2,1]).astype(int)
rfm["FrequencyScore"] = pd.qcut(rfm["구매횟수"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
rfm["MonetaryScore"] = pd.qcut(rfm["총매출"], 4, labels=[1,2,3,4]).astype(int)
rfm["RFM_Score"] = rfm["RecencyScore"].astype(int) + rfm["FrequencyScore"].astype(int) + rfm["MonetaryScore"].astype(int)

def segment(score):
    if score >= 10:
        return "VIP"
    if score >= 7:
        return "충성고객"
    if score <= 4:
        return "이탈위험"
    return "일반고객"

rfm["세그먼트"] = rfm["RFM_Score"].apply(segment)
print(rfm[["고객ID", "Recency", "구매횟수", "총매출", "세그먼트"]])

fig, ax = plt.subplots(figsize=(6,3))
rfm["세그먼트"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("고객 세그먼트")
plt.tight_layout()
plt.show()
